# ⬡ ExamGuard — AI Proctoring Server
### Run Cell 1 once. Run Cell 2 every session.
---

In [ ]:
# ═══════════════════════════════════════
# CELL 1 — Install (run once per session)
# ═══════════════════════════════════════
!pip install flask flask-cors opencv-python-headless ultralytics pillow -q
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
print('✅ All dependencies installed!')

In [ ]:
# ═══════════════════════════════════════
# CELL 2 — Start AI Server (run every session)
# ═══════════════════════════════════════
import cv2, numpy as np, base64, io, subprocess, time, threading, re
from PIL import Image
from flask import Flask, request, jsonify
from flask_cors import CORS
from ultralytics import YOLO
from datetime import datetime

# Kill old processes
subprocess.run(['fuser', '-k', '5001/tcp'], capture_output=True)
subprocess.run(['pkill', '-f', 'cloudflared'], capture_output=True)
time.sleep(2)

# ── Load Models ──
print('⏳ Loading AI models...')
face_cascade    = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
eye_cascade     = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_eye.xml')
profile_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_profileface.xml')
yolo_model      = YOLO('yolov8n.pt')

# Warm up YOLO so first request is fast
yolo_model(np.zeros((240, 320, 3), dtype=np.uint8), verbose=False)
print('✅ Models loaded and warmed up!')

# ── Config ──
BANNED_OBJECTS = {
    'cell phone' : ('phone_detected',    'HIGH',   '🚫 Phone detected in frame!'),
    'book'       : ('book_detected',     'MEDIUM', '📚 Book detected — no materials allowed!'),
    'laptop'     : ('laptop_detected',   'HIGH',   '💻 Second laptop detected!'),
    'tablet'     : ('tablet_detected',   'HIGH',   '📱 Tablet detected in frame!'),
    'remote'     : ('remote_detected',   'MEDIUM', '📡 Remote/device detected!'),
    'earphones'  : ('earphone_detected', 'MEDIUM', '🎧 Earphones detected!'),
    'headphones' : ('earphone_detected', 'MEDIUM', '🎧 Headphones detected!'),
    'paper'      : ('paper_detected',    'MEDIUM', '📄 Paper/notes detected!'),
}

stats = {'total_frames':0,'total_violations':0,'no_face_count':0,'multi_face_count':0,'object_violations':0,'start_time':datetime.now().strftime('%H:%M:%S')}

# ── Helpers ──
def decode_image(b64):
    if ',' in b64: b64 = b64.split(',')[1]
    img = Image.open(io.BytesIO(base64.b64decode(b64))).convert('RGB')
    return cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)

def detect_faces(frame):
    gray = cv2.equalizeHist(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY))
    front   = face_cascade.detectMultiScale(gray, 1.05, 4, minSize=(50, 50))
    profile = profile_cascade.detectMultiScale(gray, 1.05, 4, minSize=(50, 50))
    faces = list(front) if len(front) > 0 else []
    for pf in (profile if len(profile) > 0 else []):
        px, py, pw, ph = pf
        if not any(abs(px-fx)<60 and abs(py-fy)<60 for fx,fy,fw,fh in faces):
            faces.append(pf)
    return faces, gray

def check_eyes(gray, faces):
    if not faces: return True
    x, y, w, h = faces[0]
    eyes = eye_cascade.detectMultiScale(gray[y:y+h, x:x+w], 1.1, 4, minSize=(20,20))
    return len(eyes) >= 1

def detect_objects(frame):
    results = yolo_model(frame, verbose=False, conf=0.40, imgsz=640)
    detected = {}
    for r in results:
        for box in r.boxes:
            label = yolo_model.names[int(box.cls)]
            conf  = float(box.conf)
            if label not in detected or detected[label] < conf:
                detected[label] = round(conf * 100, 1)
    return detected

# ── Flask App ──
app = Flask(__name__)
CORS(app, resources={r'/*': {'origins': '*'}})

@app.route('/')
def index(): return jsonify({'status':'ok','message':'ExamGuard AI running!','frames':stats['total_frames']})

@app.route('/ping')
def ping(): return 'pong', 200

@app.route('/health')
def health(): return jsonify({'status':'ok','stats':stats})

@app.route('/analyze', methods=['POST'])
def analyze():
    violations, info = [], {}
    try:
        data = request.get_json(force=True, silent=True)
        if not data or not data.get('image'):
            return jsonify({'violations':[],'error':'No image'}), 400

        frame = decode_image(data['image'])
        stats['total_frames'] += 1

        # Face detection
        faces, gray = detect_faces(frame)
        count = len(faces)
        info['face_count'] = count

        if count == 0:
            violations.append({'type':'no_face','severity':'HIGH','message':'👤 No face detected — please look at the screen!'})
            stats['no_face_count'] += 1
        elif count == 1:
            if not check_eyes(gray, faces):
                violations.append({'type':'looking_away','severity':'MEDIUM','message':'👀 Looking away from screen detected!'})
        elif count == 2:
            violations.append({'type':'multiple_faces','severity':'HIGH','message':'👥 2 people detected — unauthorized person!'})
            stats['multi_face_count'] += 1
        else:
            violations.append({'type':'multiple_faces','severity':'CRITICAL','message':f'🚨 {count} people detected — exam compromised!'})
            stats['multi_face_count'] += 1

        # Object detection
        detected = detect_objects(frame)
        info['detected_objects'] = detected
        for obj, conf in detected.items():
            if obj in BANNED_OBJECTS:
                vtype, severity, message = BANNED_OBJECTS[obj]
                violations.append({'type':vtype,'severity':severity,'message':f'{message} ({conf}%)','confidence':conf})
                stats['object_violations'] += 1

        if violations: stats['total_violations'] += len(violations)
        print(f'[Frame {stats["total_frames"]}] faces={count} | violations={len(violations)} | objects={list(detected.keys())}')
        return jsonify({'violations':violations,'info':info,'frame_number':stats['total_frames']})

    except Exception as e:
        import traceback; traceback.print_exc()
        return jsonify({'violations':[],'error':str(e)}), 500

@app.route('/stats')
def get_stats(): return jsonify(stats)

@app.route('/reset', methods=['POST'])
def reset():
    stats.update({'total_frames':0,'total_violations':0,'no_face_count':0,'multi_face_count':0,'object_violations':0,'start_time':datetime.now().strftime('%H:%M:%S')})
    return jsonify({'message':'Reset done!','stats':stats})

# ── Start Flask ──
threading.Thread(target=lambda: app.run(port=5001, use_reloader=False, threaded=True), daemon=True).start()
time.sleep(2)
print('✅ Flask server running!')

# ── Start Cloudflare Tunnel ──
proc = subprocess.Popen(['./cloudflared','tunnel','--url','http://localhost:5001'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print('⏳ Starting Cloudflare tunnel...')
for line in proc.stdout:
    line = line.decode()
    match = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', line)
    if match:
        url = match.group(0)
        print(f"""
╔══════════════════════════════════════════════════╗
║       ✅  ExamGuard AI is LIVE!                  ║
╠══════════════════════════════════════════════════╣
║  🌐  {url}
╚══════════════════════════════════════════════════╝

📋  Update Render backend env var:
    AI_API = {url}

⚠️   Keep this notebook running during the exam!
        """)
        break